### Feature Engineering | Demandes d’asile UNHCR & IDMC

`Version 2:  cible +10 %, volume du segment et contexte IDMC`

### Décision métier

L'objectif ML devient : **prédire à la fin de l'année T si un segment connaîtra une hausse strictement supérieure à 10 % à T+1**.

Cette V2 ajoute une dimension d'**impact/volume** sans mélanger impact et cible :
- la cible reste une dynamique future (>10 %) ;
- le volume courant et le poids relatif du segment sont des features disponibles à T ;
- l'IDMC est un contexte de pression au **niveau pays d'origine**, et non un volume propre au corridor.

Cette séparation évite de définir artificiellement une « hausse critique » à partir d'informations futures et préserve l'interprétabilité métier.


In [28]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path("../data/consolidated/dataset_demandes_consolide_detail.csv")
OUTDIR=Path("../data/feature_engineering_outputs")
OUTDIR.mkdir(exist_ok=True)

df=pd.read_csv(DATA_PATH,low_memory=False)
TARGET="hausse_critique_demandes_t1"
THRESHOLD=0.10
SEGMENT=["coo_id","coa_id","procedure_type","app_type","dec_level","app_pc"]

assert not df.duplicated(SEGMENT+["year"]).any()
print(df.shape, df["year"].min(), df["year"].max())

print("Dataset source :", df.shape)
print("Période :", int(df["year"].min()), "→", int(df["year"].max()))

(120597, 50) 2000 2025
Dataset source : (120597, 50)
Période : 2000 → 2025


### 1. Nettoyage non destructif

Aucune ligne métier suspecte n'est supprimée. Les textes utilisés comme catégories sont standardisés (`strip`, valeurs manquantes explicites). Les anomalies quantitatives restent dans le patrimoine source et peuvent être auditées.


In [29]:
cat_source=["procedure_type","app_type","dec_level","app_pc","origin_region","asylum_region"]
for c in cat_source:
    if c in df.columns:
        df[c]=df[c].astype("string").str.strip().str.upper().fillna("INCONNU")

# applied doit être numérique ; les valeurs invalides deviennent NaN, jamais 0 silencieusement.
df["applied"]=pd.to_numeric(df["applied"],errors="coerce")
assert (df["applied"].dropna()>=0).all()


### 2. Lags exacts du volume du segment

Un lag T−1 n'est créé que si **l'année calendaire exacte T−1 existe pour le même segment**. Cela évite qu'une observation 2019 soit prise à tort comme T−1 d'une ligne 2021.

Features retenues :
- `applied` : volume courant, mesure directe de l'importance du segment ;
- `log_applied` : même information compressée pour limiter l'influence des très gros volumes ;
- `applied_lag1`, `applied_lag2` : inertie historique ;
- croissances et variations absolues passées : dynamique récente ;
- moyenne et volatilité sur 3 ans : niveau structurel et instabilité.

*Un lag correspond à une valeur passée.*


In [30]:
def add_exact_lag(base, cols, lag, keys):
    right=base[keys+["year"]+cols].copy()
    right["year"]=right["year"]+lag
    right=right.rename(columns={c:f"{c}_lag{lag}" for c in cols})
    return base.merge(right,on=keys+["year"],how="left",validate="one_to_one")

df=add_exact_lag(df,["applied"],1,SEGMENT)
df=add_exact_lag(df,["applied"],2,SEGMENT)

df["log_applied"]=np.log1p(df["applied"])
df["absolute_change_past_1"]=df["applied"]-df["applied_lag1"]
df["absolute_change_past_2"]=df["applied_lag1"]-df["applied_lag2"]
df["growth_past_1"]=np.where(df["applied_lag1"]>0,
    (df["applied"]-df["applied_lag1"])/df["applied_lag1"],np.nan)
df["growth_past_2"]=np.where(df["applied_lag2"]>0,
    (df["applied_lag1"]-df["applied_lag2"])/df["applied_lag2"],np.nan)

hist=df[["applied","applied_lag1","applied_lag2"]]
df["applied_mean_3y"]=hist.mean(axis=1,skipna=False)
df["applied_std_3y"]=hist.std(axis=1,skipna=False)
df["two_consecutive_increases"]=np.where(
    df["growth_past_1"].notna() & df["growth_past_2"].notna(),
    ((df["growth_past_1"]>0)&(df["growth_past_2"]>0)).astype(int),np.nan)


### 3. Poids relatif du segment : mesurer l'impact à T

Une hausse de 20 % sur 10 demandes n'a pas le même impact qu'une hausse de 20 % sur 10 000 demandes.

Nous ajoutons donc des parts relatives **calculées uniquement avec les données de T** :
- poids du segment dans les demandes de son pays d'origine ;
- poids du segment dans les demandes reçues par son pays d'asile.

Les dénominateurs sont calculés à `year × pays × app_pc` afin de ne pas additionner naïvement des unités de comptage différentes (personnes vs cas/familles).


In [31]:
origin_tot=(df.groupby(["year","coo_id","app_pc"],dropna=False)["applied"]
              .sum(min_count=1).rename("origin_applied_total_t").reset_index())
asylum_tot=(df.groupby(["year","coa_id","app_pc"],dropna=False)["applied"]
              .sum(min_count=1).rename("asylum_applied_total_t").reset_index())

df=df.merge(origin_tot,on=["year","coo_id","app_pc"],how="left",validate="many_to_one")
df=df.merge(asylum_tot,on=["year","coa_id","app_pc"],how="left",validate="many_to_one")

df["segment_share_origin_t"]=np.where(df["origin_applied_total_t"]>0,
                                      df["applied"]/df["origin_applied_total_t"],np.nan)
df["segment_share_asylum_t"]=np.where(df["asylum_applied_total_t"]>0,
                                      df["applied"]/df["asylum_applied_total_t"],np.nan)
df["log_origin_applied_total_t"]=np.log1p(df["origin_applied_total_t"])
df["log_asylum_applied_total_t"]=np.log1p(df["asylum_applied_total_t"])


### 4. IDMC : contexte pays d'origine, retardé d'un an

`idmc_total_origin` décrit le stock de personnes déplacées internes lié au pays d'origine. Il est donc **incorrect de le présenter comme le volume du segment origine→asile**.

La V2 construit une table unique `origine × année`, contrôle les éventuels conflits de valeurs, puis utilise `idmc_total_origin_lag1`. Le lag est effectué au niveau origine et non au niveau du segment : un nouveau corridor peut ainsi bénéficier du contexte IDMC de son pays d'origine même s'il n'existait pas à T−1.

Nous créons également un ratio `idmc_pressure_ratio_lag1 = IDMC(T−1) / demandes totales de l'origine(T−1)` pour contextualiser l'ordre de grandeur. Ce ratio reste descriptif et ne suppose aucune causalité.


In [36]:
# Audit : une origine-année ne devrait pas porter plusieurs stocks IDMC différents.
idmc_conflicts=(df.dropna(subset=["idmc_total_origin"])
                  .groupby(["coo_id","year"])["idmc_total_origin"].nunique())
print("Origine-années IDMC conflictuelles :", int((idmc_conflicts>1).sum()))

idmc_origin=(df[["coo_id","year","idmc_total_origin"]]
             .dropna(subset=["idmc_total_origin"])
             .groupby(["coo_id","year"],as_index=False)["idmc_total_origin"].max())

idmc_lag=idmc_origin.copy()
idmc_lag["year"]=idmc_lag["year"]+1
idmc_lag=idmc_lag.rename(columns={"idmc_total_origin":"idmc_total_origin_lag1"})
df=df.merge(idmc_lag,on=["coo_id","year"],how="left",validate="many_to_one")

# Total origine T-1 au même niveau de comptage app_pc.
origin_lag=origin_tot.copy()
origin_lag["year"]=origin_lag["year"]+1
origin_lag=origin_lag.rename(columns={"origin_applied_total_t":"origin_applied_total_lag1"})
df=df.merge(origin_lag,on=["year","coo_id","app_pc"],how="left",validate="many_to_one")

df["log_idmc_total_origin_lag1"]=np.log1p(df["idmc_total_origin_lag1"])
df["idmc_pressure_ratio_lag1"]=np.where(
    df["origin_applied_total_lag1"]>0,
    df["idmc_total_origin_lag1"]/df["origin_applied_total_lag1"],np.nan)


Origine-années IDMC conflictuelles : 0


### 5. Contexte décisionnel retardé et simplifié

Les décisions peuvent apporter un signal sur la pression et le traitement antérieur, mais conserver toutes leurs composantes crée de la redondance (`decisions_total`, reconnus, rejetés, clôturés...).

La V2 conserve seulement :
- `decisions_total_lag1` : intensité décisionnelle passée ;
- `protection_rate_lag1` : part des décisions accordant une forme de protection ;
- `rejection_rate_lag1` : contexte de rejet.

Les variables courantes T sont écartées par prudence sur leur calendrier de publication.


In [37]:
decision_cols=["decisions_total","decisions_recognized","decisions_other","decisions_rejected"]
df=add_exact_lag(df,decision_cols,1,SEGMENT)

den=df["decisions_total_lag1"]
df["protection_rate_lag1"]=np.where(den>0,
    (df["decisions_recognized_lag1"]+df["decisions_other_lag1"])/den,np.nan)
df["rejection_rate_lag1"]=np.where(den>0,df["decisions_rejected_lag1"]/den,np.nan)


### 6. Nouvelle cible : hausse strictement supérieure à 10 % à T+1

La cible est construite par jointure exacte sur T+1. Une ligne sans véritable année suivante reste **non labellisée** ; elle n'est jamais transformée artificiellement en classe 0.


In [38]:
future=df[SEGMENT+["year","applied"]].copy()
future["year"]=future["year"]-1
future=future.rename(columns={"applied":"applied_t1"})
df=df.merge(future,on=SEGMENT+["year"],how="left",validate="one_to_one")

df["growth_t1"]=np.where(
    df["applied"].gt(0)&df["applied_t1"].notna(),
    (df["applied_t1"]-df["applied"])/df["applied"],np.nan)

df[TARGET]=pd.Series(pd.NA,index=df.index,dtype="Int64")
mask=df["growth_t1"].notna()
df.loc[mask,TARGET]=(df.loc[mask,"growth_t1"]>THRESHOLD).astype(int)

print("Labellisables :",mask.sum(),"/",len(df),f"({mask.mean():.2%})")
display(df.loc[mask,TARGET].value_counts().sort_index().rename("n").to_frame()
        .assign(pct=lambda x:100*x["n"]/x["n"].sum()))


Labellisables : 80032 / 120597 (66.36%)


,n,pct
hausse_critique_demandes_t1,,
0,46539,58.15049
1,33493,41.84951


### 7. Sélection finale des features

### Features conservées — justification métier et ML

**Identité/contexte du segment**
- `coo_id`, `coa_id` : captent des effets structurels propres aux origines et pays d'asile ; traités comme catégories.
- `procedure_type`, `app_type`, `dec_level`, `app_pc` : décrivent le processus et l'unité de comptage.
- `origin_region`, `asylum_region` : permettent une généralisation géographique au-delà d'un pays précis.

**Temps et volume**
- `year` : capte les changements structurels globaux dans le temps.
- `applied`, `log_applied` : importance actuelle du segment et version compressée robuste aux très gros volumes.
- lags, variations, croissance, moyenne et volatilité : inertie, accélération, persistance et instabilité.

**Impact relatif**
- totaux origine/asile et parts du segment : distinguent un micro-segment d'un flux représentant une part importante de son environnement.

**Contexte retardé**
- IDMC T−1 : pression de déplacement interne au niveau origine, disponible avant T+1.
- décisions T−1 simplifiées : contexte administratif antérieur sans multiplier les variables corrélées.

### Features volontairement exclues
- identifiants techniques (`_row_id`, `_demand_row_id`) : aucune valeur métier prédictive ;
- noms/codes ISO dupliqués : redondants avec les identifiants pays ;
- variables T+1 et cible : fuite de cible ;
- solutions durables : taux de valeurs manquantes très élevés et lien trop indirect avec la cible V1 ;
- décisions courantes T : disponibilité opérationnelle à T non suffisamment garantie ;
- composantes décisionnelles laggées détaillées : redondantes avec total + taux ;
- `idmc_total_origin` courant : disponibilité à T non garantie et duplication au niveau des segments ;
- variantes françaises/anglaises de régions/pays : doublons sémantiques.


In [39]:
categorical_features=[
    "coo_id","coa_id","procedure_type","app_type","dec_level","app_pc",
    "origin_region","asylum_region"
]
numeric_features=[
    "year","applied","log_applied",
    "applied_lag1","applied_lag2",
    "absolute_change_past_1","absolute_change_past_2",
    "growth_past_1","growth_past_2","two_consecutive_increases",
    "applied_mean_3y","applied_std_3y",
    "origin_applied_total_t","asylum_applied_total_t",
    "log_origin_applied_total_t","log_asylum_applied_total_t",
    "segment_share_origin_t","segment_share_asylum_t",
    "origin_applied_total_lag1",
    "idmc_total_origin_lag1","log_idmc_total_origin_lag1","idmc_pressure_ratio_lag1",
    "decisions_total_lag1","protection_rate_lag1","rejection_rate_lag1"
]
final_features=[c for c in categorical_features+numeric_features if c in df.columns]
print("Nombre de features V2 :",len(final_features))
print(final_features)


Nombre de features V2 : 33
['coo_id', 'coa_id', 'procedure_type', 'app_type', 'dec_level', 'app_pc', 'origin_region', 'asylum_region', 'year', 'applied', 'log_applied', 'applied_lag1', 'applied_lag2', 'absolute_change_past_1', 'absolute_change_past_2', 'growth_past_1', 'growth_past_2', 'two_consecutive_increases', 'applied_mean_3y', 'applied_std_3y', 'origin_applied_total_t', 'asylum_applied_total_t', 'log_origin_applied_total_t', 'log_asylum_applied_total_t', 'segment_share_origin_t', 'segment_share_asylum_t', 'origin_applied_total_lag1', 'idmc_total_origin_lag1', 'log_idmc_total_origin_lag1', 'idmc_pressure_ratio_lag1', 'decisions_total_lag1', 'protection_rate_lag1', 'rejection_rate_lag1']


### 8. Contrôles anti-leakage et qualité

Les champs futurs sont interdits dans X. Les valeurs manquantes sont quantifiées mais non supprimées silencieusement. Les colonnes constantes sont signalées. Les parts relatives doivent rester entre 0 et 1 (hors anomalies de données).


In [40]:
forbidden={"applied_t1","growth_t1",TARGET}
assert not forbidden.intersection(final_features)

labelled=df[df[TARGET].notna()].copy()
unlabelled=df[df[TARGET].isna()].copy()
X=labelled[final_features].copy()
y=labelled[TARGET].astype(int)

assert X.index.equals(y.index)
assert ((df["segment_share_origin_t"].dropna()>=0)&(df["segment_share_origin_t"].dropna()<=1+1e-9)).all()
assert ((df["segment_share_asylum_t"].dropna()>=0)&(df["segment_share_asylum_t"].dropna()<=1+1e-9)).all()

missing=(X.isna().mean()*100).sort_values(ascending=False).rename("missing_pct").to_frame()
constant=[c for c in X if X[c].nunique(dropna=False)<=1]
print("Colonnes constantes :",constant)
display(missing)


Colonnes constantes : []


,missing_pct
idmc_pressure_ratio_lag1,59.842313
log_idmc_total_origin_lag1,59.839814
idmc_total_origin_lag1,59.839814
growth_past_2,40.230158
applied_std_3y,40.230158
applied_mean_3y,40.230158
two_consecutive_increases,40.230158
absolute_change_past_2,40.230158
applied_lag2,34.954768
protection_rate_lag1,33.136745


### 9. Journal de sélection et exports

`DROP` signifie **exclu du modèle**, jamais supprimé du patrimoine source. Le journal permet d'auditer chaque décision.


In [42]:
status={}
reason={}
for c in df.columns:
    if c in final_features:
        status[c]="KEEP"
        reason[c]="Feature V2 retenue : disponible à T et pertinente pour volume, dynamique, contexte ou segmentation."
    elif c in {"applied_t1","growth_t1",TARGET}:
        status[c]="DROP_LEAKAGE"; reason[c]="Information future ou cible."
    elif c in {"_row_id","_demand_row_id","decisions_source_rows","solutions_source_rows"}:
        status[c]="DROP_TECHNICAL"; reason[c]="Traçabilité technique, sans valeur métier prédictive."
    elif c in {"returned_refugees","resettlement","naturalisation","returned_idps","has_solutions_data"}:
        status[c]="DROP_MISSINGNESS"; reason[c]="Très forte incomplétude et lien indirect avec la cible V1."
    elif c in {"idmc_total_origin","has_idmc_data"}:
        status[c]="DROP_T0_UNCERTAIN"; reason[c]="Version courante non garantie à T ; version IDMC T-1 préférée."
    elif c.startswith("decisions_") or c in {"recognition_rate","rejection_rate","has_decisions_data"}:
        status[c]="DROP_REDUNDANT_OR_T0"; reason[c]="Décisions courantes ou composantes détaillées redondantes ; contexte synthétique T-1 préféré."
    else:
        status[c]="DROP_REDUNDANT_OR_UNUSED"; reason[c]="Doublon descriptif, référentiel ou feature non nécessaire à la V2."

log=pd.DataFrame({"feature":list(status),"decision":[status[c] for c in status],
                  "justification":[reason[c] for c in status]})

ml_export=labelled[final_features+[TARGET]].copy()
ml_export.to_csv(OUTDIR/"dataset_ml_features.csv",index=False)
unlabelled[final_features+["applied_t1","growth_t1",TARGET]].to_csv(
    OUTDIR/"rows_unlabelled.csv",index=False)
log.to_csv(OUTDIR/"feature_selection_log.csv",index=False)

print("Exports :")
for p in OUTDIR.iterdir(): print("-",p)
display(log[log["decision"]=="KEEP"])


Exports :
- ..\data\feature_engineering_outputs\dataset_ml_features.csv
- ..\data\feature_engineering_outputs\feature_selection_log.csv
- ..\data\feature_engineering_outputs\rows_unlabelled.csv


,feature,decision,justification
1,year,KEEP,Feature V2 retenue : disponible à T et pertine...
2,coo_id,KEEP,Feature V2 retenue : disponible à T et pertine...
6,coa_id,KEEP,Feature V2 retenue : disponible à T et pertine...
10,procedure_type,KEEP,Feature V2 retenue : disponible à T et pertine...
11,app_type,KEEP,Feature V2 retenue : disponible à T et pertine...
12,dec_level,KEEP,Feature V2 retenue : disponible à T et pertine...
13,app_pc,KEEP,Feature V2 retenue : disponible à T et pertine...
14,applied,KEEP,Feature V2 retenue : disponible à T et pertine...
36,origin_region,KEEP,Feature V2 retenue : disponible à T et pertine...
44,asylum_region,KEEP,Feature V2 retenue : disponible à T et pertine...
